In [2]:
# ==============================================================================
# CELL 1: SETUP & DEPENDENCIES
# ==============================================================================
!pip install rasterio scikit-learn torch matplotlib pandas numpy scipy pyproj pvl planetaryimage

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from pyproj import Transformer
from google.colab import drive, files
from scipy.ndimage import distance_transform_edt, uniform_filter, gaussian_filter
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Set random seeds for exact reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Set Drive path (Auto-created if it doesn't exist)
DRIVE_FOLDER = '/content/drive/MyDrive/LUNAR_SITE/'
os.makedirs(DRIVE_FOLDER, exist_ok=True)

# Update with ALL available file names
DEM_FILE = os.path.join(DRIVE_FOLDER, "LDEM_80S_80MPP_ADJ.tiff")
LDSM_FILE = os.path.join(DRIVE_FOLDER, "LDSM_80S_80MPP_ADJ.tiff")
LDEC_FILE = os.path.join(DRIVE_FOLDER, "LDEC_80S_80MPP_ADJ.tiff")
SUN_VIS_FILE = os.path.join(DRIVE_FOLDER, "AVGVISIB_85S_060M_201608.IMG")
EARTH_VIS_FILE = os.path.join(DRIVE_FOLDER, "AVGVISIB_85S_060M_201608_EARTH.IMG")
PSR_FILE = os.path.join(DRIVE_FOLDER, "LPSR_85S_060M_201608.IMG")

print("Environment setup & Google Drive datasets mapped.")

  Using cached pvl-1.3.2-py2.py3-none-any.whl.metadata (25 kB)
  Using cached planetaryimage-0.5.0-py2.py3-none-any.whl.metadata (4.4 kB)
Using cached pvl-1.3.2-py2.py3-none-any.whl (66 kB)
Using cached planetaryimage-0.5.0-py2.py3-none-any.whl (13 kB)
Mounted at /content/drive
Environment setup & Google Drive datasets mapped.


In [3]:
# ==============================================================================
# CELL 2: ENHANCED GEOSPATIAL FEATURE EXTRACTION
# ==============================================================================
def read_raster(filepath, out_shape, fallback_func):
    """Helper function to load a raster or generate a synthetic fallback."""
    if os.path.exists(filepath):
        with rasterio.open(filepath) as src:
            return src.read(1, out_shape=out_shape)
    else:
        print(f"Warning: {os.path.basename(filepath)} not found. Using fallback.")
        return fallback_func()

def load_lunar_rasters(dem_path, ldsm_path, ldec_path, sun_vis_path, earth_vis_path, psr_path, grid_size=(1000, 1000)):
    print("--- Loading NASA Geospatial Datasets ---")

    # 1. Elevation (DEM)
    if os.path.exists(dem_path):
        with rasterio.open(dem_path) as src:
            dem = src.read(1, out_shape=grid_size)
            res_m = src.res[0] if src.res[0] > 0 else 80.0
    else:
        dem = np.random.normal(loc=-1500, scale=800, size=grid_size)
        res_m = 80.0

    # 2. Derive/Load Slope & Roughness
    gy, gx = np.gradient(dem, res_m)
    fallback_slope = np.degrees(np.arctan(np.sqrt(gx**2 + gy**2)))
    slope = read_raster(ldsm_path, grid_size, lambda: fallback_slope)

    mean_dem = uniform_filter(dem, size=5)
    roughness = np.sqrt(np.maximum(0, uniform_filter(dem**2, size=5) - mean_dem**2))
    prominence = dem - gaussian_filter(dem, sigma=10)

    # 3. Density/Counts (LDEC)
    ldec = read_raster(ldec_path, grid_size, lambda: np.ones(grid_size) * 100)

    # 4. Visibility (Sun & Earth)
    sun_vis = read_raster(sun_vis_path, grid_size, lambda: np.clip(100 - slope * 1.5, 0, 100))
    earth_vis = read_raster(earth_vis_path, grid_size, lambda: np.clip(sun_vis - 10, 0, 100))

    # 5. PSR Map & Proximity (km)
    psr_data = read_raster(psr_path, grid_size, lambda: (sun_vis < 5).astype(int))
    psr_mask = (psr_data > 0).astype(int)

    dist_to_psr_px = distance_transform_edt(1 - psr_mask)
    dist_to_psr_km = (dist_to_psr_px * res_m) / 1000.0

    return {
        'dem': dem,
        'slope': slope,
        'roughness': roughness,
        'prominence': prominence,
        'sun_visibility': sun_vis,
        'earth_visibility': earth_vis,
        'ldec_density': ldec,
        'psr_mask': psr_mask,
        'psr_dist_km': dist_to_psr_km,
        'resolution_m': res_m,
        'shape': grid_size
    }

def calculate_physics_informed_targets(data):
    # Feature Normalizations
    norm_sun = data['sun_visibility'] / 100.0
    norm_earth = data['earth_visibility'] / 100.0
    norm_slope = np.clip(data['slope'] / 45.0, 0, 1)
    norm_rough = np.clip(data['roughness'] / np.percentile(data['roughness'], 95), 0, 1)

    p_min, p_max = np.percentile(data['prominence'], [5, 95])
    norm_prom = np.clip((data['prominence'] - p_min) / (p_max - p_min + 1e-6), 0, 1)

    psr_close = np.exp(-((data['psr_dist_km'] - 3.0) ** 2) / (2 * (3.0 ** 2)))
    psr_direct = np.exp(-(data['psr_dist_km'] ** 2) / (2 * (1.5 ** 2)))

    # Base Targets (Modified to use both Sun & Earth vis where applicable)
    score_hab = (0.30 * norm_sun + 0.10 * norm_earth + 0.25 * (1 - norm_slope) + 0.25 * psr_close + 0.10 * (1 - norm_rough)) * 100.0
    score_min = (0.35 * norm_slope + 0.35 * norm_rough + 0.30 * psr_direct) * 100.0
    score_land = (0.50 * (1 - norm_slope) + 0.35 * (1 - norm_rough) + 0.15 * norm_sun) * 100.0
    score_solar = (0.55 * norm_sun + 0.25 * (1 - norm_slope) + 0.12 * norm_prom + 0.08 * (1 - norm_rough)) * 100.0
    score_comms = (0.35 * norm_prom + 0.25 * norm_earth + 0.20 * norm_sun + 0.10 * (1 - norm_slope) + 0.10 * (1 - norm_rough)) * 100.0
    score_sci = (0.30 * norm_slope + 0.30 * psr_direct + 0.25 * norm_prom + 0.15 * norm_rough) * 100.0

    # NEW: Ice Proxy & Radiation Safety (Modeled as a "Safe" Zone to keep scores consistently "higher = better")
    ice_proxy = np.where(data['psr_mask'] == 1, 100.0, 100.0 * np.exp(-data['psr_dist_km'] / 1.5))
    radiation_risk = (0.7 * norm_sun + 0.3 * (1 - norm_prom)) * 100.0
    radiation_safety = 100.0 - radiation_risk

    return score_hab, score_min, score_land, score_solar, score_comms, score_sci, ice_proxy, radiation_safety

lunar_data = load_lunar_rasters(DEM_FILE, LDSM_FILE, LDEC_FILE, SUN_VIS_FILE, EARTH_VIS_FILE, PSR_FILE)
targets = calculate_physics_informed_targets(lunar_data)

--- Loading NASA Geospatial Datasets ---


In [4]:
# ==============================================================================
# CELL 3: TRAIN / VAL / TEST SPLIT & ADVANCED DEEP LEARNING (8 TARGETS)
# ==============================================================================
# Flatten inputs (Now 8 features)
X = np.column_stack([
    lunar_data['dem'].ravel(),
    lunar_data['slope'].ravel(),
    lunar_data['roughness'].ravel(),
    lunar_data['prominence'].ravel(),
    lunar_data['sun_visibility'].ravel(),
    lunar_data['earth_visibility'].ravel(),
    lunar_data['ldec_density'].ravel(),
    lunar_data['psr_dist_km'].ravel()
])

# 8 Targets
Y = np.column_stack([t.ravel() for t in targets])

# Splits
X_train, X_temp, Y_train, Y_temp = train_test_split(X, Y, test_size=0.30, random_state=42)
X_val, X_test, Y_val, Y_test = train_test_split(X_temp, Y_temp, test_size=0.50, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print("--- Training Random Forest Regressor ---")
rf = RandomForestRegressor(n_estimators=100, max_depth=14, random_state=42, n_jobs=-1)
rf.fit(X_train_s, Y_train)
rf_test_preds = rf.predict(X_test_s)
print(f"Random Forest Test R² Score: {r2_score(Y_test, rf_test_preds):.4f}")

class EarlyStopping:
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

class LunarSiteDeepNet(nn.Module):
    def __init__(self, in_features, out_features):
        super(LunarSiteDeepNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Linear(64, out_features)
        )

    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dl_model = LunarSiteDeepNet(in_features=8, out_features=8).to(device)

train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train_s), torch.FloatTensor(Y_train)), batch_size=2048, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.FloatTensor(X_val_s), torch.FloatTensor(Y_val)), batch_size=2048, shuffle=False)

criterion = nn.MSELoss()
optimizer = optim.AdamW(dl_model.parameters(), lr=0.003, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
early_stopping = EarlyStopping(patience=6, min_delta=1e-3)

EPOCHS = 50
print(f"\n--- Training PyTorch Deep Model on [{device}] with Early Stopping & AdamW ---")

for epoch in range(EPOCHS):
    dl_model.train()
    train_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(dl_model(bx), by)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * bx.size(0)
    train_loss /= len(train_loader.dataset)

    dl_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for vx, vy in val_loader:
            vx, vy = vx.to(device), vy.to(device)
            val_loss += criterion(dl_model(vx), vy).item() * vx.size(0)
    val_loss /= len(val_loader.dataset)

    scheduler.step(val_loss)
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    early_stopping(val_loss)
    if early_stopping.early_stop:
        print(f"\n>>> Early Stopping triggered at Epoch {epoch+1}.")
        break

--- Training Random Forest Regressor ---
Random Forest Test R² Score: 0.9990

--- Training PyTorch Deep Model on [cpu] with Early Stopping & AdamW ---
Epoch 01/50 | Train Loss: 1562.8185 | Val Loss: 41.2817
Epoch 02/50 | Train Loss: 8.9595 | Val Loss: 1.0574
Epoch 03/50 | Train Loss: 0.9291 | Val Loss: 0.4013
Epoch 04/50 | Train Loss: 0.5416 | Val Loss: 0.2189
Epoch 05/50 | Train Loss: 0.3933 | Val Loss: 0.1359
Epoch 06/50 | Train Loss: 0.3149 | Val Loss: 0.0940
Epoch 07/50 | Train Loss: 0.2568 | Val Loss: 0.0617
Epoch 08/50 | Train Loss: 0.2178 | Val Loss: 0.0533
Epoch 09/50 | Train Loss: 0.1977 | Val Loss: 0.0435
Epoch 10/50 | Train Loss: 0.1846 | Val Loss: 0.0420
Epoch 11/50 | Train Loss: 0.1715 | Val Loss: 0.0346
Epoch 12/50 | Train Loss: 0.1625 | Val Loss: 0.0376
Epoch 13/50 | Train Loss: 0.1543 | Val Loss: 0.0418
Epoch 14/50 | Train Loss: 0.1462 | Val Loss: 0.0352
Epoch 15/50 | Train Loss: 0.1356 | Val Loss: 0.0270
Epoch 16/50 | Train Loss: 0.1340 | Val Loss: 0.0283
Epoch 17/50 |

In [5]:
# ==============================================================================
# CELL 4: SITE EXTRACTION (10 KM SPATIAL SEPARATION FILTER & LAT/LON)
# ==============================================================================
X_full_s = scaler.transform(X)
full_preds = rf.predict(X_full_s)
grid_shape = lunar_data['shape']

use_case_names = [
    "Habitation", "Mineral Research", "Landing Site",
    "Solar Energy Generation", "Communication & Navigation", "Scientific Research",
    "Ice Extraction Proxy", "High Radiation Safety Zone"
]

pred_maps = [full_preds[:, i].reshape(grid_shape) for i in range(8)]

def extract_ranked_sites(score_map, use_case_label, lunar_data, dem_path, top_k=5, min_dist_km=10.0):
    res_m = lunar_data['resolution_m']
    min_dist_px = (min_dist_km * 1000.0) / res_m
    flat_indices = np.argsort(score_map.ravel())[::-1]

    selected_sites = []
    selected_coords_px = []

    has_crs = False
    if os.path.exists(dem_path):
        with rasterio.open(dem_path) as src:
            if src.crs is not None:
                has_crs = True
                transform = src.transform
                transformer = Transformer.from_crs(src.crs, "EPSG:4326", always_xy=True)

    for idx in flat_indices:
        r, c = np.unravel_index(idx, score_map.shape)

        too_close = any(np.sqrt((r - sr)**2 + (c - sc)**2) < min_dist_px for sr, sc in selected_coords_px)

        if not too_close:
            selected_coords_px.append((r, c))

            if has_crs:
                x, y = rasterio.transform.xy(transform, r, c)
                lon, lat = transformer.transform(x, y)
            else:
                center_r, center_c = score_map.shape[0] / 2.0, score_map.shape[1] / 2.0
                dx = (c - center_c) * res_m
                dy = (center_r - r) * res_m
                dist_m = np.sqrt(dx**2 + dy**2)
                lat = -90.0 + (dist_m / 1737400.0) * (180.0 / np.pi)
                lon = np.degrees(np.arctan2(dx, dy))

            # Retrieve dynamic Ice/Risk values based on array index
            ice_val = targets[6].reshape(grid_shape)[r, c]
            rad_safety = targets[7].reshape(grid_shape)[r, c]
            rad_risk = 100.0 - rad_safety

            selected_sites.append({
                'Use Case': use_case_label,
                'Rank': len(selected_sites) + 1,
                'Latitude (°N/S)': round(float(lat), 4),
                'Longitude (°E)': round(float(lon), 4),
                'Elevation (m)': round(float(lunar_data['dem'][r, c]), 2),
                'Slope (deg)': round(float(lunar_data['slope'][r, c]), 2),
                'Sun Vis (%)': round(float(lunar_data['sun_visibility'][r, c]), 2),
                'Earth Vis (%)': round(float(lunar_data['earth_visibility'][r, c]), 2),
                'Dist to PSR (km)': round(float(lunar_data['psr_dist_km'][r, c]), 2),
                'Ice Proxy Score': round(float(ice_val), 2),
                'Radiation Risk': round(float(rad_risk), 2),
                'Suitability Score': round(float(score_map[r, c]), 2)
            })

            if len(selected_sites) == top_k:
                break

    return pd.DataFrame(selected_sites)

all_use_cases_dfs = []
for name, p_map in zip(use_case_names, pred_maps):
    df_uc = extract_ranked_sites(p_map, name, lunar_data, DEM_FILE, top_k=5, min_dist_km=10.0)
    all_use_cases_dfs.append(df_uc)
    print(f"\n" + "="*85 + f"\n TOP 5 {name.upper()} SITES (>=10 km Separation)\n" + "="*85)
    print(df_uc.to_string(index=False))

# Combine into master dataframe
master_df = pd.concat(all_use_cases_dfs, ignore_index=True)


 TOP 5 HABITATION SITES (>=10 km Separation)
  Use Case  Rank  Latitude (°N/S)  Longitude (°E)  Elevation (m)  Slope (deg)  Sun Vis (%)  Earth Vis (%)  Dist to PSR (km)  Ice Proxy Score  Radiation Risk  Suitability Score
Habitation     1         -89.3768         52.7420       -1324.67         0.35        99.48          89.48              0.08            94.81           82.80              81.85
Habitation     2         -89.0731         -9.9985       -2281.87         0.15        99.78          89.78              0.08            94.81           93.77              80.97
Habitation     3         -88.8010       -115.2682       -1560.09         1.63        97.55          87.55              0.08            94.81           84.00              80.31
Habitation     4         -88.7448         98.9476        -662.50         3.14        95.29          85.29              0.08            94.81           72.33              79.70
Habitation     5         -89.8395         64.6986       -1544.37         3

In [6]:
# ==============================================================================
# CELL 5: SAVE TO DRIVE & TRIGGER LOCAL CSV DOWNLOAD
# ==============================================================================


# 2. Trigger instant local file download in browser
local_filename = "ranked_lunar_sites_comprehensive.csv"
master_df.to_csv(local_filename, index=False)
files.download(local_filename)
print(f"[✓] Browser download initiated for: {local_filename}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[✓] Browser download initiated for: ranked_lunar_sites_comprehensive.csv
